# Compute per-site band quality baseline
* Downloads (if not already present) the 30 min low-tide, 0% cloud satellite images per site across 2016-2026
* Computes the per-site, per-band median/MAD baseline (see sentinel2.compute_and_save_band_quality_baseline)
* Only needs to be run once - the baseline is cached to utils.get_band_quality_baseline_path() and reused by sentinel2.flag_anomalous_band_dates

In [1]:
import geopandas
import pathlib
import dask.distributed

module_path = pathlib.Path.cwd().parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2

%load_ext autoreload
%autoreload 2

In [2]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:61161/status,
Dashboard: http://127.0.0.1:61161/status,Workers: 4
Total threads: 8,Total memory: 31.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:61162,Workers: 0
Dashboard: http://127.0.0.1:61161/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:61190,Total threads: 2
Dashboard: http://127.0.0.1:61191/status,Memory: 7.93 GiB
Nanny: tcp://127.0.0.1:61165,


# Values to edit

In [3]:
all_training_sites =  ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]

In [4]:
baseline_sites = all_training_sites
baseline_tide_deltas = {"30mins": {"hrs": 0, "mins": 30},
                        "45mins": {"hrs": 0, "mins": 45},
                        "1hr": {"hrs": 1, "mins": 0}}
baseline_max_cloud_cover = 0
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

In [5]:
data_path = utils.get_data_path()
utils.create_data_folders()

# Cells to run
* Download the low-tide no-cloud images by year and site (skips sites already downloaded)
* Compute and save the per-site, per-band quality baseline

### Download satellite images
The 0% cloud, low-tide images across sites for each year and tide delta (30min/45min/1hr),
reusing the prediction-style per-year storage. Pooling across tide deltas (rather than dropping
the low-tide requirement entirely) grows the per-site sample size while keeping the baseline
restricted to genuinely low-tide reflectance.

In [ ]:
satellite_paths = {}
for tide_delta_name, tide_delta in baseline_tide_deltas.items():
    predictions_path, satellite_path = utils.get_prediction_path(
        sample_method="sampling_2", method_2_threshold=0.5,
        model_max_cloud_cover=baseline_max_cloud_cover,
        predict_max_cloud_cover=baseline_max_cloud_cover,
        predict_low_tide_delta_hrs=tide_delta["hrs"],
        predict_low_tide_delta_mins=tide_delta["mins"],
        model_low_tide_delta_hrs=tide_delta["hrs"], # dummy value as models not considered this notebook
        model_low_tide_delta_mins=tide_delta["mins"]) # dummy value as models not considered this notebook
    satellite_paths[tide_delta_name] = satellite_path

In [ ]:
for tide_delta_name, tide_delta in baseline_tide_deltas.items():
    predictions_path, satellite_path = utils.get_prediction_path(
        sample_method="sampling_2", method_2_threshold=0.5,
        model_max_cloud_cover=baseline_max_cloud_cover,
        predict_max_cloud_cover=baseline_max_cloud_cover,
        predict_low_tide_delta_hrs=tide_delta["hrs"],
        predict_low_tide_delta_mins=tide_delta["mins"],
        model_low_tide_delta_hrs=tide_delta["hrs"], # dummy value as models not considered this notebook
        model_low_tide_delta_mins=tide_delta["mins"]) # dummy value as models not considered this notebook

    for baseline_site in baseline_sites:
        for year in years:
            print(f"{tide_delta_name} {baseline_site}: year {year}")

            satellite_file = satellite_path / f"{baseline_site}_{year}_sentinel-2.nc"
            site_polygon_file = utils.get_site_polygon_path(baseline_site)

            if (satellite_path / f"{baseline_site}_{years[-1]}_sentinel-2.nc").exists():
                print(f"\tAlready run for {baseline_site} as {baseline_site}_{years[-1]}_sentinel-2.nc exists. "
                      "Delete if you want redownloaded.")
                break

            if not satellite_file.exists():
                print(f"\tSave satellite image of the site {baseline_site} around lowtide without cloud in the year {year}")
                site_polygon = geopandas.read_file(site_polygon_file)
                satellite_data = sentinel2.get_low_tide_no_cloud_images_in_year(
                    geometry=site_polygon,
                    max_cloud_cover=baseline_max_cloud_cover,
                    year=year,
                    low_tide_delta_hrs=tide_delta["hrs"],
                    low_tide_delta_mins=tide_delta["mins"],
                )
                if len(satellite_data['time']) == 0:
                    print("Warning: No satellite data without cloud. Ignore")
                else:
                    print("\t\tSave")
                    utils.write_netcdf_conventions_in_place(satellite_data)
                    satellite_data = satellite_data.rio.reproject(utils.CRS_NZTM)
                    satellite_data = satellite_data.rio.clip(site_polygon.geometry, drop=True, all_touched=True)
                    utils.write_netcdf_conventions_in_place(satellite_data)
                    utils.save_netcdf(satellite_data, satellite_file)

30mins Ihutai: year 2016
	Already run for Ihutai as Ihutai_2026_sentinel-2.nc exists. Delete if you want redownloaded.
30mins CatlinsLake: year 2016
30mins CatlinsLake: year 2017
	Save satellite image of the site CatlinsLake around lowtide without cloud in the year 2017
	Time from low tide: 0:29:58.974000 for S2A_MSIL2A_20170626T223701_R072_T59GLJ_20210210T015542
	Time from low tide: 0:07:28.974000 for S2A_MSIL2A_20171120T222531_R029_T59GLJ_20201014T210953
	Time from low tide: 0:20:00.973000 for S2B_MSIL2A_20170810T223659_R072_T59GLJ_20210210T223634
	Time from low tide: 0:13:41.026000 for S2A_MSIL2A_20171021T222541_R029_T59GLJ_20201015T011921
	Time from low tide: 0:26:28.974000 for S2A_MSIL2A_20171220T222531_R029_T59GLJ_20201019T193033
	Time from low tide: 0:12:30.973000 for S2B_MSIL2A_20171205T222529_R029_T59GLJ_20201014T121240
	Time from low tide: 0:12:30.973000 for S2B_MSIL2A_20171205T222529_R029_T59GLJ_20201014T121240
	Low tide tiles: 6 from a total lowish cloud cover tiles of 104


### Compute and save the band quality baseline
Pools all downloaded years across all tide deltas per site into one dataset, then computes
per-site, per-band median/MAD and caches it to utils.get_band_quality_baseline_path().

In [ ]:
baseline = sentinel2.compute_and_save_band_quality_baseline(
    training_sites=baseline_sites,
    satellite_images_paths=list(satellite_paths.values()),
)
baseline

In [ ]:
import pandas
pandas.read_csv(r"C:\Local\repos\seagrass-detection\data\band_quality_baseline_old.csv")